In [3]:
"""
End-to-end training entry point.

Steps:
  1. Seed RNGs (numpy / torch / cuda) so a training run is reproducible.
  2. Discover training/validation tiles and compute per-band normalisation
     statistics on the training split only — the same stats are reused for
     validation and at inference time (stored alongside the model in MLflow).
  3. Build Albumentations transforms (resize → flip → normalize → ToTensorV2).
  4. Hand the dataloaders to `pl.Trainer.fit`. Lightning drives the optimiser,
     scheduler, early-stopping and checkpointing.

Run from the project root with `python -m src.train` (or
`uv run python -m src.train`) so the `src.*` package imports resolve.
"""

import os
import random
import numpy as np
import torch
import pytorch_lightning as pl

from torch.utils.data import DataLoader, random_split
from torch import Generator
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

from src.data.loading import load_data
from src.data.normalization import compute_global_normalization
from src.data.transforms import build_transform
from src.training.lightning import get_lightning_module
from src.data.dataset import SegmentationDataset


# ==========================================================
# 1️⃣ CONFIG
# ==========================================================

CONFIG = {
    "train_regions": [
        "AT332",
        "LU000",
    ],
    "train_years": ["2018"],
    "test_regions": ["LU000"],
    "test_year": "2021",
    "batch_size": 32,
    "test_batch_size": 16,
    "epochs": 20,
    "lr": 1e-3,
    "n_bands": 14,
    "resize": 512,
    "num_workers": os.cpu_count(),
    "seed": 42,
}


# ==========================================================
# 2️⃣ UTILS
# ==========================================================


def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    pl.seed_everything(seed, workers=True)


def build_region_year_list(regions, years):
    return [f"{r}_{y}" for r in regions for y in years]


# ==========================================================
# 3️⃣ MAIN PIPELINE
# ==========================================================


# Reproducibilité
set_seed(CONFIG["seed"])

# -------- Dataset IDs --------
train_ids = build_region_year_list(
    CONFIG["train_regions"],
    CONFIG["train_years"],
)

test_ids = build_region_year_list(
    CONFIG["test_regions"],
    [CONFIG["test_year"]],
)

# -------- Normalisation --------
print("📊 Computing normalization...")
mean, std = compute_global_normalization(train_ids, CONFIG["n_bands"])



Seed set to 42


📊 Computing normalization...


In [5]:
train_ids

['AT332_2018', 'LU000_2018']

In [6]:
# -------- Chargement --------
print("📂 Loading data...")
train_patches, train_labels = load_data(train_ids)
test_patches, test_labels = load_data(test_ids)



📂 Loading data...
reading https://minio.lab.sspcloud.fr/projet-funathon/2026/project3/data/images/LU000/2021/filename2bbox.parquet




In [7]:
# -------- Transforms --------
train_transform = build_transform(mean, std, augment=True, resize=CONFIG["resize"])

test_transform = build_transform(mean, std, augment=False, resize=CONFIG["resize"])



In [10]:
full_dataset = SegmentationDataset(
    patchs=train_patches,
    labels=train_labels,
    n_bands=CONFIG["n_bands"],
    from_s3=False,
    transform=train_transform,
)

test_dataset = SegmentationDataset(
    patchs=test_patches,
    labels=test_labels,
    n_bands=CONFIG["n_bands"],
    from_s3=False,
    transform=test_transform,
)

# -------- Split train / val --------
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=Generator().manual_seed(CONFIG["seed"]),
)

In [38]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["test_batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
)

In [ ]:
from transformers import AutoModelForSemanticSegmentation
import requests


id2label = requests.get(
    "https://minio.lab.sspcloud.fr/projet-funathon/2026/project3/data/clcplus-backbone-id2label.json"
).json()
id2label = {int(k): v for k, v in id2label.items()}
label2id = {v: k for k, v in id2label.items()}


model_id = "nvidia/mit-b5"
model = AutoModelForSemanticSegmentation.from_pretrained(
    model_id,
    num_channels = 14,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

config.json: 0.00B [00:00, ?B/s]

[transformers] You passed `num_labels=12` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin:   0%|          | 0.00/328M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1156 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b5
Key                                              | Status     |                                                                                                  
-------------------------------------------------+------------+--------------------------------------------------------------------------------------------------
classifier.weight                                | UNEXPECTED |                                                                                                  
classifier.bias                                  | UNEXPECTED |                                                                                                  
decode_head.linear_c.{0, 1, 2, 3}.proj.weight    | MISSING    |                                                                                                  
decode_head.batch_norm.num_batches_tracked       | MISSING    |                                               

In [27]:
train_dataset


In [32]:
len(test_dataset.patchs)

351

In [35]:
train_loader.dataset

In [50]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./segmentation-demo",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    save_strategy="no",  # Don't save, just demo
    logging_steps=1,
    do_train=True,
    do_eval=True,
)

def compute_metrics(eval_pred):
    """Optional: computes example accuracy."""
    logits, labels = eval_pred
    # logits: (batch, num_labels, H, W), labels: (batch, H, W)
    predictions = logits.argmax(1)
    acc = (predictions == labels).float().mean().item()
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_loader.dataset,
    eval_dataset=val_loader.dataset,
    compute_metrics=compute_metrics,  # optional but recommended
)

In [ ]:
trainer.train()

/home/onyxia/work/funathon-project3/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/home/onyxia/work/funathon-project3/.venv/lib/python3.13/site-packages/osgeo/gdal.py:330: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Step,Training Loss
